In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- precompute_cross_join ---
def _top_n_stub(input_df, query_d3_document_id, pairwise_metric, n):
    return [query_d3_document_id] * n

FIX_PRECOMPUTE_CROSS_JOIN_FIND_TOP_N_MATCHES_SINGLE_DOCUMENT = _top_n_stub
FIX_PRECOMPUTE_CROSS_JOIN_INPUT_DF_PD = pd.DataFrame({"scores": [1,2,3]}, index=pd.Index([1,2,3], name="document_id"))
FIX_PRECOMPUTE_CROSS_JOIN_INPUT_DF_PL = pl.DataFrame({"document_id": [1,2,3], "scores": [1,2,3]})
FIX_PRECOMPUTE_CROSS_JOIN_PAIRWISE_METRIC = lambda a, b: float(np.dot(a, b))
FIX_PRECOMPUTE_CROSS_JOIN_QUERY_D3_DOCUMENT_ID = 0
FIX_PRECOMPUTE_CROSS_JOIN_NEW_DF_PD = pd.DataFrame({"document_id": [1,2,3], "scores": [1,2,3]})
FIX_PRECOMPUTE_CROSS_JOIN_NEW_DF_PL = pl.from_pandas(FIX_PRECOMPUTE_CROSS_JOIN_NEW_DF_PD)

# --- precompute_id_cross_frame ---
FIX_PRECOMPUTE_ID_CROSS_FRAME_QUERY_D3_DOCUMENT_ID = 0
FIX_PRECOMPUTE_ID_CROSS_FRAME_INPUT_DF_PD = pd.DataFrame({"id":[1,2,3],"value":[10,20,30]})
FIX_PRECOMPUTE_ID_CROSS_FRAME_INPUT_DF_PL = pl.from_pandas(
    FIX_PRECOMPUTE_ID_CROSS_FRAME_INPUT_DF_PD.reset_index()
).with_columns(
    pl.col("index").alias("d3_document_id")
)

pd.Series.progress_apply = pd.Series.apply
n = 2
print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_precompute_cross_join(find_top_n_matches_single_document, input_df, pairwise_metric, query_d3_document_id, new_df):
    return (
        pd.DataFrame(data=input_df.index, columns=["document_id"])
        .assign(
            scores=lambda new_df: new_df["document_id"].progress_apply(
                lambda query_d3_document_id: find_top_n_matches_single_document(
                    input_df, query_d3_document_id, pairwise_metric, n
                )
            )
        )
        .set_index("document_id")
    )
    return None

def before_precompute_id_cross_frame(query_d3_document_id, input_df):
    for d3_document_id in input_df.index:
        if d3_document_id == query_d3_document_id:
            continue
        ...
    return None

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_precompute_cross_join(find_top_n_matches_single_document, input_df, pairwise_metric, query_d3_document_id, new_df):
    return (
        pl.DataFrame({"document_id": input_df.get_column("document_id")})
        .with_columns(
            pl.col("document_id").map_elements(
                lambda query_d3_document_id: find_top_n_matches_single_document(
                    input_df, query_d3_document_id, pairwise_metric, n
                ),
                return_dtype=pl.Object,
            ).alias("scores")
        )
    )
    return None

def gen_precompute_id_cross_frame(query_d3_document_id, input_df):
    pd = pl  # LLM used `import polars as pd`
    if "index" not in input_df.columns:
        input_df = input_df.with_row_index("index")

    for d3_document_id in input_df["index"]:
        if d3_document_id == query_d3_document_id:
            continue
        ...
    return None

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def _comparison_label(label):
    text = str(label)
    if text.lstrip().startswith(("L2", "L3")):
        return text
    return f"L2 equivalence {text}"

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: precompute_cross_join ===

# L1 smoke – generated
try:
    _r = gen_precompute_cross_join(FIX_PRECOMPUTE_CROSS_JOIN_FIND_TOP_N_MATCHES_SINGLE_DOCUMENT, FIX_PRECOMPUTE_CROSS_JOIN_INPUT_DF_PL, FIX_PRECOMPUTE_CROSS_JOIN_PAIRWISE_METRIC, FIX_PRECOMPUTE_CROSS_JOIN_QUERY_D3_DOCUMENT_ID, FIX_PRECOMPUTE_CROSS_JOIN_NEW_DF_PL)
    print("✅ L1 smoke gen_precompute_cross_join: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_precompute_cross_join: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_precompute_cross_join(FIX_PRECOMPUTE_CROSS_JOIN_FIND_TOP_N_MATCHES_SINGLE_DOCUMENT, FIX_PRECOMPUTE_CROSS_JOIN_INPUT_DF_PD, FIX_PRECOMPUTE_CROSS_JOIN_PAIRWISE_METRIC, FIX_PRECOMPUTE_CROSS_JOIN_QUERY_D3_DOCUMENT_ID, FIX_PRECOMPUTE_CROSS_JOIN_NEW_DF_PD)
    print("✅ L1 smoke before_precompute_cross_join: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_precompute_cross_join: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_precompute_cross_join(FIX_PRECOMPUTE_CROSS_JOIN_FIND_TOP_N_MATCHES_SINGLE_DOCUMENT, FIX_PRECOMPUTE_CROSS_JOIN_INPUT_DF_PD, FIX_PRECOMPUTE_CROSS_JOIN_PAIRWISE_METRIC, FIX_PRECOMPUTE_CROSS_JOIN_QUERY_D3_DOCUMENT_ID, FIX_PRECOMPUTE_CROSS_JOIN_NEW_DF_PD).reset_index()
    _rg = gen_precompute_cross_join(FIX_PRECOMPUTE_CROSS_JOIN_FIND_TOP_N_MATCHES_SINGLE_DOCUMENT, FIX_PRECOMPUTE_CROSS_JOIN_INPUT_DF_PL, FIX_PRECOMPUTE_CROSS_JOIN_PAIRWISE_METRIC, FIX_PRECOMPUTE_CROSS_JOIN_QUERY_D3_DOCUMENT_ID, FIX_PRECOMPUTE_CROSS_JOIN_NEW_DF_PL)
    compare(_rb, _rg, "precompute_cross_join", check_row_order=True)
except Exception as _e:
    print(f"❌ L2 equivalence precompute_cross_join: setup error — {type(_e).__name__}: {_e}")

# L3 edge - compare empty-input behaviour with the oracle.
try:
    _rb=before_precompute_cross_join(FIX_PRECOMPUTE_CROSS_JOIN_FIND_TOP_N_MATCHES_SINGLE_DOCUMENT, FIX_PRECOMPUTE_CROSS_JOIN_INPUT_DF_PD.head(0), FIX_PRECOMPUTE_CROSS_JOIN_PAIRWISE_METRIC, FIX_PRECOMPUTE_CROSS_JOIN_QUERY_D3_DOCUMENT_ID, FIX_PRECOMPUTE_CROSS_JOIN_NEW_DF_PD.head(0)); _rg=gen_precompute_cross_join(FIX_PRECOMPUTE_CROSS_JOIN_FIND_TOP_N_MATCHES_SINGLE_DOCUMENT, FIX_PRECOMPUTE_CROSS_JOIN_INPUT_DF_PL.head(0), FIX_PRECOMPUTE_CROSS_JOIN_PAIRWISE_METRIC, FIX_PRECOMPUTE_CROSS_JOIN_QUERY_D3_DOCUMENT_ID, FIX_PRECOMPUTE_CROSS_JOIN_NEW_DF_PL.head(0)); compare(_rb,_rg,"L3 edge precompute_cross_join empty input",check_row_order=True)
except Exception as _e: print(f"❌ L3 edge precompute_cross_join: {type(_e).__name__}: {_e}")
